In [ ]:
from preprocessing import build_sequences, split
from train import fine_tune
seqs = build_sequences('token')
from baseline_if import run_isolation_forest 

In [4]:
train, test = split(seqs)
fine_tune('token', train, rank=16) 

  Train (normal only): 57
  Test  (mixed):       114  (57 normal, 57 anomalous)

  Loading base model...


Loading weights: 100%|██████████| 288/288 [00:10<00:00, 27.91it/s]


trainable params: 6,389,760 || all params: 2,620,731,648 || trainable%: 0.2438
  Preparing dataset (57 sequences)...


Map: 100%|██████████| 57/57 [00:00<00:00, 136.40 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


  Training token (rank=16, epochs=3)...


c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.527784
10,2.071098


c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*ar

  Saved adapter to checkpoints\token\r16


WindowsPath('checkpoints/token/r16')

In [ ]:

model, tok = load_finetuned('token', rank=16)
scores, labels = score_sequences(test, model, tok)
evaluate(scores, labels, model_name='llm_lora', service='token')

Loading weights: 100%|██████████| 288/288 [00:11<00:00, 25.51it/s]


    Scored 10/114  (last: 1.9783, label=0)
    Scored 20/114  (last: 2.0409, label=0)
    Scored 30/114  (last: 2.2135, label=1)
    Scored 40/114  (last: 1.8714, label=0)
    Scored 50/114  (last: 2.0774, label=1)
    Scored 60/114  (last: 1.9082, label=1)
    Scored 70/114  (last: 1.8448, label=1)
    Scored 80/114  (last: 1.9634, label=0)
    Scored 90/114  (last: 1.8384, label=1)
    Scored 100/114  (last: 2.0724, label=0)
    Scored 110/114  (last: 1.9679, label=0)
  [llm_lora | token] AUCROC=0.5337  F1=0.6746  thresh=1.1679  n_test=114


{'service': 'token',
 'model': 'llm_lora',
 'aucroc': 0.5337,
 'f1': 0.6746,
 'threshold': np.float64(1.1679),
 'n_test': 114,
 'n_anomaly': 57}

In [ ]:
if_res = run_isolation_forest(train_seqs, test_seqs)

In [2]:
import torch, json
from preprocessing import build_sequences, split, SERVICES
from train import fine_tune, LORA_RANK
from score import load_finetuned, score_sequences
from evaluate import evaluate, save_results

In [ ]:
RESULTS = []

for service in SERVICES:
    print(f"\n{'='*55}\n  Service: {service}\n{'='*55}")

    # ── Data ─────────────────────────────────────────────────────────────────
    seqs = build_sequences(service)
    if not seqs:
        print("  No data — skipping.")
        continue
    train_seqs, test_seqs = split(seqs)

    # ── Isolation Forest baseline ─────────────────────────────────────────────
    print("\n  [Isolation Forest]")
    if_res = run_isolation_forest(train_seqs, test_seqs)
    if if_res:
        RESULTS.append({**if_res, "service": service, "model": "isolation_forest"})

    # ── LLM + LoRA ────────────────────────────────────────────────────────────
    print("\n  [LLM + LoRA]")
    fine_tune(service, train_seqs, rank=LORA_RANK)

    model, tokenizer = load_finetuned(service, rank=LORA_RANK)
    scores, labels   = score_sequences(test_seqs, model, tokenizer)
    llm_res          = evaluate(scores, labels,
                                model_name="llm_lora", service=service)
    if llm_res:
        RESULTS.append(llm_res)

    del model; torch.cuda.empty_cache()

# ── Save & display ────────────────────────────────────────────────────────────
save_results(RESULTS)